# Precios dinámicos y gestión de ingresos

## Tipo de problema

Ajustar tarifas en tiempo real (aerolíneas, hoteles, VTCs) según la demanda anticipada para maximizar el ingreso total por asiento o habitación disponible.

## Formas de resolverlo

Los nombres de la primera columna coinciden con la [tabla del README](../README.md#formas-de-resolverlo).

| Enfoque (README) | Aplicación en este problema | Uso en este ejercicio |
|---|---|---|
| Programación dinámica | Elegir tarifas considerando inventario y oportunidades futuras. | **Principal** |
| No lineal o cuadrática | Optimizar precios con funciones de demanda no lineales. | Alternativa |
| Modelos estocásticos y simulación | Evaluar políticas de precios bajo escenarios de demanda. | Alternativa |

**Enfoque utilizado:** Programación dinámica.

**Técnica concreta:** Ecuación de Bellman por periodo y habitaciones restantes, con probabilidades de compra dadas.

## Ejercicio propuesto

Vender dos habitaciones para una misma noche en tres periodos de reserva.  
En cada periodo hay una oportunidad de venta de una habitación. Tarifas:  
30, 50 o 70 €. Probabilidades de compra por periodo y tarifa:  
(0.9, 0.6, 0.3), (0.8, 0.5, 0.2), (0.7, 0.4, 0.1).  
Sin cancelaciones ni valor residual. Se conoce si se vendió antes de decidir  
la siguiente tarifa. Objetivo: ingreso esperado máximo.  
Método exacto: programación dinámica (Bellman) por periodo y stock.

## Método elegido para este ejercicio

**Enfoque utilizado:** Programación dinámica.

Programación dinámica: el estado combina periodo y habitaciones restantes. Para cada tarifa se ponderan los casos de venta y de no venta, incluyendo el valor de conservar la habitación para el futuro.

## Implementación

Ejecuta las celdas de código en orden. Solo se utiliza la biblioteca estándar de Python.

In [1]:
from functools import lru_cache


def resolver():
    precios = (30, 50, 70)
    probabilidades = ((0.9, 0.6, 0.3), (0.8, 0.5, 0.2), (0.7, 0.4, 0.1))
    politica = {}

    @lru_cache(None)
    def valor(periodo, stock):
        if periodo == len(probabilidades) or stock == 0:
            return 0.0
        opciones = []
        for precio, p in zip(precios, probabilidades[periodo]):
            ingreso = p * (precio + valor(periodo + 1, stock - 1))
            ingreso += (1 - p) * valor(periodo + 1, stock)
            opciones.append((ingreso, precio))
        ingreso, precio = max(opciones)
        politica[periodo, stock] = precio
        return ingreso

    esperado = valor(0, 2)
    return esperado, politica

## Resolución y resultados

In [2]:
esperado, politica = resolver()
print(f'Ingreso esperado óptimo con 2 habitaciones: {esperado:.2f} €')
for (periodo, stock), precio in sorted(politica.items()):
    print(f'Periodo {periodo + 1}, quedan {stock}: ofrecer {precio} €')

Ingreso esperado óptimo con 2 habitaciones: 69.70 €
Periodo 1, quedan 2: ofrecer 50 €
Periodo 2, quedan 1: ofrecer 50 €
Periodo 2, quedan 2: ofrecer 50 €
Periodo 3, quedan 1: ofrecer 30 €
Periodo 3, quedan 2: ofrecer 30 €


## Interpretación

Mientras quede stock, ofrecer **50 €, 50 € y 30 €** en los tres periodos. El ingreso esperado máximo con dos habitaciones es **69,70 €**. Es una media probabilística: el ingreso realizado puede ser distinto.

**Alcance:** los datos son sintéticos. Las garantías y supuestos se refieren a esta instancia; al aumentar su tamaño, la enumeración puede resultar costosa.